# Chapter 3: Final Model Evaluation, Paired Bootstrapping & Deployment

This workbook conducts the final evaluation on the unseen test set ($25\%$ stratified split) and establishes statistical confidence intervals via bootstrapping ($B = 1000$ iterations). We systematically compare four distinct modeling paradigms:
1. **Heuristic Baseline (No-ML Algorithm):** A deterministic classifier that assigns entities to the macro-region based only on the middle of the minimal 5-hour inactivity window location.
2. **Multinomial Logistic Regression:** A standardized linear baseline (`StandardScaler` + `LogisticRegression`) weighted inversely to class frequencies.
3. **XGBoost 1 (Unpenalized Baseline Architecture):** XGBoost trained without explicit train-validation divergence penalties.
4. **XGBoost 2 (Regularized / Penalized Architecture):** The final selected gradient boosted architecture selected via the overfitting-penalized Brier score objective ($\lambda = 0.5$).

**Key Pipeline Stages:**
1. **Environment & MLflow Tracking:** Initializes analytical dependencies and connects to the `btc_timezone_analysis_model_results` experiment.
2. **Probabilistic & Selective Classification Metrics:** Defines diagnostic functions for test-set Confusion Matrices, class-wise Calibration Curves, Brier Scores, and Area Under the Risk-Coverage Curve (AURC).
3. **Data Ingestion & Baseline Feature Selection:** Loads the preprocessed entity matrix and isolates the 24-hour diurnal feature vector.
4. **Heuristic Baseline Model (Non-ML Circular Centroid Classifier):** Computes empirical circular sleep centroids per region and evaluates deterministic nearest-centroid assignment.
5. **Single-Split Model Evaluations:** Evaluates Logistic Regression, XGBoost 1, and XGBoost 2 across a single holdout split.
6. **Paired Bootstrap Hypothesis Testing:** Evaluates Logistic Regression, XGBoost 1, and XGBoost 2 on a fixed holdout test set using 1,000 shared bootstrap index resamples to compute exact $95\%$ confidence intervals ($2.5\%$, $50\%$, $97.5\%$ quantiles).
7. **Model Deployment:** Exports the final `XGBClassifier` and `LabelEncoder` into `timezone_model.pkl` for the Streamlit web application.

### 1: Environment Initialization & MLflow Tracking Setup

Initializes core scientific computing, machine learning, and directional statistics (scipy.stats.circmean) libraries, and sets the active MLflow experiment for test-set result logging.

In [16]:
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, accuracy_score, brier_score_loss, auc
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.stats import circmean

import xgboost as xgb

# Configure Pandas display options
pd.set_option('display.max_rows', 1000)

In [17]:
# Connect to local MLflow server and set the dedicated results experiment
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("btc_timezone_analysis_model_results")

<Experiment: artifact_location='/home/matej/btc_timezone_analysis/03_model_training/mlruns_artifacts/2', creation_time=1789896748946, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789896748946, lifecycle_stage='active', name='btc_timezone_analysis_model_results', tags={}, trace_location=None, workspace='default'>

### 2: Probabilistic & Selective Classification Metrics

Defines evaluation utilities adapted specifically for comparing Training vs. Holdout Test performance, including text-annotated classification reports alongside normalized confusion matrices, quantile-binned calibration curves, and selective Risk-Coverage (AURC) integration.

In [18]:
def class_report(y_train, y_train_pred, y_test, y_test_pred, target_name):
    """Generates a 2x2 figure combining text Classification Reports and normalized Confusion Matrices for Train and Test sets."""
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 12), 
                             gridspec_kw={'height_ratios': [0.5, 1.5]})
    
    # 1. Training Classification Report (Text Panel)
    axes[0, 0].text(-0.2, 0.4, classification_report(y_train, y_train_pred, target_names=["America", "Central_Asia", "East_Asia_Pac", "Euro_Africa"]), 
                    fontsize=13, family='monospace', va='center')
    axes[0, 0].set_title("Classification Report (Train)", fontsize=14, fontweight='bold')
    axes[0, 0].axis('off') 
    
    # 2. Testing Classification Report (Text Panel)
    axes[0, 1].text(-0.2, 0.4, classification_report(y_test, y_test_pred, target_names=["America", "Central_Asia", "East_Asia_Pac", "Euro_Africa"]), 
                    fontsize=13, family='monospace', va='center')
    axes[0, 1].set_title("Classification Report (Test)", fontsize=14, fontweight='bold')
    axes[0, 1].axis('off') 
    
    # 3. Training Confusion Matrix (Row-Normalized)
    ConfusionMatrixDisplay.from_predictions(
        y_train, y_train_pred, 
        ax=axes[1, 0], cmap='Blues', normalize='true',
        labels=target_name
    )
    axes[1, 0].set_title("Training Confusion Matrix (Normalized)")
    axes[1, 0].tick_params(axis='x', rotation=90)
    
    # 4. Testing Confusion Matrix (Row-Normalized)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_test_pred, 
        ax=axes[1, 1], cmap='Greens', normalize='true',
        labels=target_name
    )
    axes[1, 1].set_title("Testing Confusion Matrix (Normalized)")
    axes[1, 1].tick_params(axis='x', rotation=90)
    
    plt.savefig("./plots/class_report.png", dpi=300, bbox_inches='tight')
    plt.close()

In [19]:
def calibration_curve_classes(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred, colors, labels):
    """Generates class-wise reliability diagrams (Calibration Curves) for Train and Test sets."""
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    for j, a in zip(range(2), ["train", "test"]):
        ax[j].plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")
        ax[j].plot([0.75, 0.75], [-0.05, 1.05], "--", color="#C0C0C0")
        
        for l, c in zip(labels, colors):
            proba = locals()[f"y_{a}_proba"]
            pred  = locals()[f"y_{a}_pred"]
            true  = locals()[f"y_{a}"]
        
            subset = (true == l)
            
            confidences = np.max(proba[subset], axis=1)
            accuracies = (pred[subset] == true[subset]).astype(int)
            
            # Partition predictions into 5 quantile-based confidence bins
            n_bins = 5
            percentiles = np.linspace(0.0, 1.0, n_bins + 1)
            bins = np.quantile(confidences, percentiles)
            bin_indices = np.digitize(confidences, bins) - 1
            bin_indices[bin_indices == n_bins] = n_bins - 1
            
            bin_accuracies, bin_confidences, bin_percentages = [], [], []
            
            for i in range(n_bins):
                mask = (bin_indices == i)
                if np.sum(mask) > 0: # Ensure bin contains samples
                    bin_accuracies.append(np.mean(accuracies[mask]))
                    bin_confidences.append(np.mean(confidences[mask]))
                    bin_percentages.append((np.sum(mask) / len(confidences)) * 100)
        
            ax[j].plot(bin_confidences, bin_accuracies, marker='o', linestyle='-', color=c, label=l)
        
        ax[j].set_xlabel("Confidence")
        ax[j].set_ylabel("True Accuracy")
        ax[j].set_title(f"Calibration Graph per each class on {a} set")
        ax[j].legend(loc="upper left")
        ax[j].grid(True, alpha=0.5)
        ax[j].set_xlim([-0.05, 1.05])
        ax[j].set_ylim([-0.05, 1.05])
        
    plt.savefig("./plots/calibration_graph.png", dpi=300, bbox_inches='tight')
    plt.close()

In [20]:
def calculate_aurc(y_true, y_prob, y_pred):
    """Calculates the Area Under the Risk-Coverage Curve (AURC)."""
    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)
    
    confidences = np.max(y_prob, axis=1)
    errors = (y_true_np != y_pred_np).astype(int)
    
    # Sort instances by descending confidence
    sorted_indices = np.argsort(-confidences)
    sorted_errors = errors[sorted_indices]
    
    n_samples = len(y_true_np)
    coverages = np.arange(1, n_samples + 1) / n_samples
    
    # Compute cumulative selective risk
    cumulative_errors = np.cumsum(sorted_errors)
    risks = cumulative_errors / np.arange(1, n_samples + 1)
    
    coverages = np.insert(coverages, 0, 0.0)
    risks = np.insert(risks, 0, risks[0]) 
    
    aurc_score = auc(coverages, risks)
    return coverages, risks, aurc_score

In [21]:
def get_mlflow_confidence_metrics(
    y_test_true, y_test_pred, y_test_prob, 
    y_train_true, y_train_pred, y_train_prob, 
    target_names
):
    """Computes Brier scores, overall accuracy, AURC, and selective coverage accuracies on the Test set."""
    metrics = {}

    # 1. Global Brier Scores and Overall Accuracy
    metrics['train_brier_score'] = float(brier_score_loss(label_binarize(y_train_true, classes=target_names), y_train_prob))
    metrics['test_brier_score'] = float(brier_score_loss(label_binarize(y_test_true, classes=target_names), y_test_prob))
    metrics['train_accuracy'] = float(accuracy_score(y_train_true, y_train_pred))
    metrics['test_accuracy'] = float(accuracy_score(y_test_true, y_test_pred))
    metrics['samples'] = len(y_test_true)

    fold_metrics_collection = {
        "test_coverage_10_accuracy": [], 
        "test_coverage_25_accuracy": [], 
        "test_coverage_50_accuracy": [], 
        "test_aurc": [],
        "train_aurc": []
    }
    
    n_splits = 5
    test_fold_size = len(y_test_true) // n_splits
    train_fold_size = len(y_train_true) // n_splits
    
    for i in range(n_splits):
        # 1. Test Slice
        test_start = i * test_fold_size
        test_end = (i + 1) * test_fold_size if i < n_splits - 1 else len(y_test_true)
        
        y_test_true_fold = np.array(y_test_true[test_start:test_end])
        y_test_pred_fold = np.array(y_test_pred[test_start:test_end])
        y_test_prob_fold = np.array(y_test_prob[test_start:test_end])
        
        # 2. Train Slice
        train_start = i * train_fold_size
        train_end = (i + 1) * train_fold_size if i < n_splits - 1 else len(y_train_true)
        
        y_train_true_fold = np.array(y_train_true[train_start:train_end])
        y_train_pred_fold = np.array(y_train_pred[train_start:train_end])
        y_train_prob_fold = np.array(y_train_prob[train_start:train_end])

        # 3. Test Coverage Metrics (Top 10%, 25%, 50% most confident predictions)
        confidences = np.max(y_test_prob_fold, axis=1)
        sorted_idx = np.argsort(confidences)[::-1]
        y_true_sorted = y_test_true_fold[sorted_idx]
        y_pred_sorted = y_test_pred_fold[sorted_idx]
        
        coverages = [0.10, 0.25, 0.50]
        for c in coverages:
            k = max(1, int(c * len(y_test_true_fold))) 
            top_true = y_true_sorted[:k]
            top_pred = y_pred_sorted[:k]
            
            c_str = f"test_coverage_{int(c * 100)}"
            fold_metrics_collection[f"{c_str}_accuracy"].append(float(accuracy_score(top_true, top_pred)))
            
        # 4. Test & Train AURC Integration
        _, _, test_fold_aurc = calculate_aurc(y_test_true_fold, y_test_prob_fold, y_test_pred_fold)
        fold_metrics_collection['test_aurc'].append(float(test_fold_aurc))
        
        _, _, train_fold_aurc = calculate_aurc(y_train_true_fold, y_train_prob_fold, y_train_pred_fold)
        fold_metrics_collection['train_aurc'].append(float(train_fold_aurc))
        
    for key, values in fold_metrics_collection.items():
        metrics[key] = float(np.mean(values))
    
    return metrics

In [22]:
def plot_risk_coverage_curve(y_train, y_train_proba, y_train_pred, y_test, y_test_proba, y_test_pred):
    """Plots the Risk-Coverage curves for Train and Test sets side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    datasets = [
        ("Train", y_train, y_train_proba, y_train_pred, axes[0]),
        ("Test", y_test, y_test_proba, y_test_pred, axes[1])
    ]
    
    for name, true, proba, pred, ax in datasets:
        coverages, risks, aurc_score = calculate_aurc(true, proba, pred)
        
        ax.plot(coverages, risks, color='blue', linewidth=2, label=f'Model AURC = {aurc_score:.4f}')
        
        baseline_error = np.mean(np.array(true) != np.array(pred))
        ax.axhline(y=baseline_error, color='red', linestyle='--', label=f'Full Dataset Error ({baseline_error:.4f})')
        
        ax.set_title(f"Risk-Coverage Curve ({name} Set)", fontsize=14, fontweight='bold')
        ax.set_xlabel("Coverage (Proportion of dataset evaluated)")
        ax.set_ylabel("Risk (Error rate on covered samples)")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.5)
        ax.set_xlim([-0.05, 1.05])
        
    plt.savefig("./plots/risk_coverage.png", dpi=300, bbox_inches="tight")
    plt.close()

### 3: Data Ingestion & Baseline Feature Selection
Loads the preprocessed entity matrix and restricts the primary feature vector (col_names) to the 24-hour relative diurnal distribution ("0" through "23"), which empirical cross-validation proved to be optimal against temporal subdivision sparsity.

In [23]:
# Load the final preprocessed dataset
data = pd.read_csv("/home/matej/btc_timezone_analysis/02_data_preprocessing/database_from_bitcointalk.csv")
len(data)

1934

In [24]:
data.columns = data.columns.astype(str)
data.head()

,Unnamed: 0,entity,offset,is_regional_location,0,1,2,3,4,5,...,21_weekend_y,22_weekday_y,22_weekend_y,23_weekday_y,23_weekend_y,total_weekday_y,total_weekend_y,log_total_weekday_y,log_total_weekend_y,dst_shift_y
0,0,Entity 100,7.0,0,0.020921,0.037657,0.025105,0.050209,0.050209,0.058577,...,0.023256,0.005650,0.011628,0.011299,0.023256,153.0,62.0,5.030438,4.127134,0.0
1,1,Entity 1000,2.0,0,0.107143,0.107143,0.071429,0.035714,0.017857,0.035714,...,0.033333,0.020000,0.033333,0.040000,0.033333,26.0,6.0,3.258097,1.791759,11.0
2,2,Entity 1001,2.0,0,0.034483,0.034483,0.051724,0.051724,0.068966,0.068966,...,0.038462,0.017857,0.038462,0.017857,0.038462,32.0,2.0,3.465736,0.693147,-4.0
3,3,Entity 1002,-5.0,0,0.053571,0.071429,0.017857,0.017857,0.017857,0.017857,...,0.028571,0.022222,0.057143,0.044444,0.142857,21.0,11.0,3.044522,2.397895,2.0
4,4,Entity 1004,7.0,0,0.017241,0.017241,0.017241,0.051724,0.086207,0.068966,...,0.034483,0.018868,0.034483,0.018868,0.034483,29.0,5.0,3.367296,1.609438,12.0


In [25]:
# Isolate the 24-hour diurnal distribution features and target macro-region
X = data
y = data["region"]
col_names = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23"]

### 4: Heuristic Baseline Model (Non-ML Circular Centroid Classifier)

To quantify the value added by machine learning, we first evaluate a deterministic domain-heuristic baseline. For each macro-region in the training set, we compute the circular mean (scipy.stats.circmean) of the 5-hour minimum inactivity midpoint (min_5h_activity) on $[0, 24)$. Unseen test entities are assigned to the region minimizing the shortest circular distance on the 24-hour clock.

In [26]:
def calibrate_baseline(midpoints, y_train):
    """Calculates the empirical UTC sleep centroid for each macro-region using circular mean."""
    centroids = {}
    
    for region in np.unique(y_train):
        region_data = midpoints[y_train == region]
        # circmean properly handles the 24-hour midnight boundary wrap-around
        centroids[region] = circmean(region_data, high=24, low=0)
        
    return centroids

def predict_baseline(midpoints, centroids):
    """Assigns entities to the macro-region with the closest circular sleep centroid."""
    predictions = []
    regions = list(centroids.keys())
    
    for user_mid in midpoints:
        best_region = None
        min_dist = float('inf')
        
        for region in regions:
            reg_mid = centroids[region]
            
            # Compute shortest angular distance on a 24-hour clock
            dist = abs(user_mid - reg_mid)
            circ_dist = min(dist, 24 - dist)
            
            if circ_dist < min_dist:
                min_dist = circ_dist
                best_region = region
                
        predictions.append(best_region)
        
    return np.array(predictions)

In [32]:
# Stratified Train-Test Split (Fixed seed = 1019 for reproducibility)
num = 1019
X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=num)

# Calibrate regional sleep centroids on Training data and predict on Holdout Test data
regional_centroids = calibrate_baseline(X_train_full['min_5h_activity'], y_train)
print(regional_centroids)

y_train_pred = predict_baseline(X_train_full['min_5h_activity'], regional_centroids)
y_test_pred = predict_baseline(X_test_full['min_5h_activity'], regional_centroids)

{'America': np.float64(8.94621896029762), 'Central_South_Asia': np.float64(22.241308612619118), 'East_Asia_Pac': np.float64(19.804945081626393), 'Euro_Africa': np.float64(3.2134751595546467)}


In [33]:
# Evaluate and log Heuristic Baseline results to MLflow
target_names = ["America", "Euro_Africa", "Central_South_Asia", "East_Asia_Pac"]
class_report(y_train, y_train_pred, y_test, y_test_pred, target_names)
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

with mlflow.start_run(run_name="baseline_model"):
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)

    mlflow.set_tag("model_type", "baseline model")
    mlflow.set_tag("data type", "5h inactivity window")

    mlflow.log_artifact("./plots/class_report.png", artifact_path="plots")

🏃 View run baseline_model at: http://127.0.0.1:5000/#/experiments/2/runs/e1708ffe6c2e46cfa4088b8388ccd951
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### 5: Single-Split Model Evaluations
In this phase, we evaluate the Multinomial Logistic Regression baseline alongside two configurations of the XGBoost classifier (unpenalized and penalized) on the fixed $25\%$ holdout test split (`random_state=1019`).

In [36]:
target_names = ["America", "Central_South_Asia", "East_Asia_Pac", "Euro_Africa"]
colors = ['red', 'green', 'blue', 'black']

X_train = X_train_full[col_names]
X_test = X_test_full[col_names]

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_encoded)

y_train_true = np.array(y_train)
y_test_true = np.array(y_test)

In [37]:
# =========================================================
# 1. Logistic Regression (Train & Single-Split Evaluation)
# =========================================================
with mlflow.start_run(run_name="logistic_regression_model"):
    mlflow.set_tag("model_type", "logistic regression test")
    mlflow.set_tag("data type", "24 hour transaction distribution")
    
    lr_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('log_reg', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=num))
    ])
    lr_pipeline.fit(X_train, y_train)

    y_train_pred_lr = np.array(lr_pipeline.predict(X_train))
    y_train_proba_lr = np.array(lr_pipeline.predict_proba(X_train))

    y_test_pred_lr = np.array(lr_pipeline.predict(X_test))
    y_test_proba_lr = np.array(lr_pipeline.predict_proba(X_test))

    metrics = get_mlflow_confidence_metrics(
        y_test_true, y_test_pred_lr, y_test_proba_lr, 
        y_train_true, y_train_pred_lr, y_train_proba_lr, 
        lr_pipeline.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train_true, y_train_pred_lr, y_test_true, y_test_pred_lr, target_names)
    calibration_curve_classes(y_train_true, y_train_proba_lr, y_train_pred_lr, y_test_true, y_test_proba_lr, y_test_pred_lr, colors, target_names)
    plot_risk_coverage_curve(y_train_true, y_train_proba_lr, y_train_pred_lr, y_test_true, y_test_proba_lr, y_test_pred_lr)
    mlflow.log_artifact("./plots/class_report.png", artifact_path="plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path="plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path="plots")


# =========================================================
# 2. XGBoost 1 - Unpenalized (Train & Single-Split Evaluation)
# =========================================================
with mlflow.start_run(run_name="xgboost_model_1"):
    mlflow.set_tag("model_type", "xgboost one model")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    xgb1 = xgb.XGBClassifier(
        n_estimators=751,
        eta=0.01798824047992439,
        gamma=1.2524473330203283,
        reg_lambda=5.284962184665449,
        max_depth=8,
        min_child_weight=4,
        subsample=0.7416097742897796,
        colsample_bytree=0.8670211875166212,
        random_state=num
    )
    xgb1.fit(X_train, y_train_encoded, sample_weight=sample_weights)

    y_train_pred_xgb1 = np.array(le.inverse_transform(xgb1.predict(X_train)))
    y_train_proba_xgb1 = np.array(xgb1.predict_proba(X_train))

    y_test_pred_xgb1 = np.array(le.inverse_transform(xgb1.predict(X_test)))
    y_test_proba_xgb1 = np.array(xgb1.predict_proba(X_test))

    metrics = get_mlflow_confidence_metrics(
        y_test_true, y_test_pred_xgb1, y_test_proba_xgb1, 
        y_train_true, y_train_pred_xgb1, y_train_proba_xgb1, 
        xgb1.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train_true, y_train_pred_xgb1, y_test_true, y_test_pred_xgb1, target_names)
    calibration_curve_classes(y_train_true, y_train_proba_xgb1, y_train_pred_xgb1, y_test_true, y_test_proba_xgb1, y_test_pred_xgb1, colors, target_names)
    plot_risk_coverage_curve(y_train_true, y_train_proba_xgb1, y_train_pred_xgb1, y_test_true, y_test_proba_xgb1, y_test_pred_xgb1)
    mlflow.log_artifact("./plots/class_report.png", artifact_path="plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path="plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path="plots")


# =========================================================
# 3. XGBoost 2 - Penalized (Train & Single-Split Evaluation)
# =========================================================
with mlflow.start_run(run_name="xgboost_model_2"):
    mlflow.set_tag("model_type", "xgboost one model")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    xgb2 = xgb.XGBClassifier(
        n_estimators=777,
        eta=0.05564936741180356,
        gamma=5.724469359582323,
        reg_lambda=1.0005545286124224,
        max_depth=5,
        min_child_weight=13,
        subsample=0.6649481827307877,
        colsample_bytree=0.9093752666296373,
        random_state=num
    )
    xgb2.fit(X_train, y_train_encoded, sample_weight=sample_weights)

    y_train_pred_xgb2 = np.array(le.inverse_transform(xgb2.predict(X_train)))
    y_train_proba_xgb2 = np.array(xgb2.predict_proba(X_train))

    y_test_pred_xgb2 = np.array(le.inverse_transform(xgb2.predict(X_test)))
    y_test_proba_xgb2 = np.array(xgb2.predict_proba(X_test))

    metrics = get_mlflow_confidence_metrics(
        y_test_true, y_test_pred_xgb2, y_test_proba_xgb2, 
        y_train_true, y_train_pred_xgb2, y_train_proba_xgb2, 
        xgb2.classes_
    )
    mlflow.log_metrics(metrics)
    class_report(y_train_true, y_train_pred_xgb2, y_test_true, y_test_pred_xgb2, target_names)
    calibration_curve_classes(y_train_true, y_train_proba_xgb2, y_train_pred_xgb2, y_test_true, y_test_proba_xgb2, y_test_pred_xgb2, colors, target_names)
    plot_risk_coverage_curve(y_train_true, y_train_proba_xgb2, y_train_pred_xgb2, y_test_true, y_test_proba_xgb2, y_test_pred_xgb2)
    mlflow.log_artifact("./plots/class_report.png", artifact_path="plots")
    mlflow.log_artifact("./plots/calibration_graph.png", artifact_path="plots")
    mlflow.log_artifact("./plots/risk_coverage.png", artifact_path="plots")

🏃 View run logistic_regression_model at: http://127.0.0.1:5000/#/experiments/2/runs/14e51f6ff5b34e1088093963803e8d47
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run xgboost_model_1 at: http://127.0.0.1:5000/#/experiments/2/runs/1405c55c00614e63bb96aff61fba1901
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run xgboost_model_2 at: http://127.0.0.1:5000/#/experiments/2/runs/8bb0349088cb4885adfdae19efca4c24
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### 6: Paired Bootstrap Hypothesis Testing

Using the predictions from the models trained in Section 5, we evaluate all three classifiers across 1,000 identical bootstrap resamples (`shared_bootstrap_indices`) drawn with replacement exclusively from the unseen test set. This allows us to compute exact $95\%$ confidence intervals for all probabilistic metrics without retraining.

In [39]:
n_iterations = 1000
random_seed = num

# ---------------------------------------------------------
# 1. Generate Shared Bootstrap Indices on the Holdout Test Set
# ---------------------------------------------------------
test_len = len(y_test_true)
rng = np.random.default_rng(seed=random_seed)
shared_bootstrap_indices = [
    rng.choice(test_len, size=test_len, replace=True) 
    for _ in range(n_iterations)
]


# =========================================================
# 2. LOGISTIC REGRESSION (Paired Bootstrap)
# =========================================================
with mlflow.start_run(run_name="logistic_regression_paired"):
    mlflow.set_tag("model_type", "logistic regression paired bootstrapped")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    all_metrics = []
    all_class_reports = []

    for boot_indices in shared_bootstrap_indices:
        y_test_boot = y_test_true[boot_indices]
        y_test_pred_boot = y_test_pred_lr[boot_indices]
        y_test_proba_boot = y_test_proba_lr[boot_indices]

        metrics = get_mlflow_confidence_metrics(
            y_test_boot, y_test_pred_boot, y_test_proba_boot, 
            y_train_true, y_train_pred_lr, y_train_proba_lr, 
            lr_pipeline.classes_
        )
        all_metrics.append(metrics)

        report_dict = classification_report(
            y_test_boot, y_test_pred_boot, target_names=target_names, 
            output_dict=True, zero_division=0
        )
        
        flat_report = {}
        for label, class_metrics in report_dict.items():
            if isinstance(class_metrics, dict):
                for metric_name, val in class_metrics.items():
                    flat_report[f"{label}_{metric_name}"] = val
            else:
                flat_report[f"{label}"] = class_metrics
        all_class_reports.append(flat_report)
    
    pd.DataFrame(all_metrics).quantile([0.025, 0.50, 0.975]).T.to_csv("./plots/lr_overall_metrics_quantiles.csv")
    pd.DataFrame(all_class_reports).quantile([0.025, 0.50, 0.975]).T.to_csv("./plots/lr_classification_report_quantiles.csv")
    mlflow.log_artifact("./plots/lr_overall_metrics_quantiles.csv", artifact_path="metrics")
    mlflow.log_artifact("./plots/lr_classification_report_quantiles.csv", artifact_path="metrics")


# =========================================================
# 3. XGBOOST 1 - Unpenalized (Paired Bootstrap)
# =========================================================
with mlflow.start_run(run_name="xgboost_1_paired"):
    mlflow.set_tag("model_type", "xgboost 1 paired bootstrapped")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    all_metrics_1 = []
    all_class_reports_1 = []

    for boot_indices in shared_bootstrap_indices:
        y_test_boot = y_test_true[boot_indices]
        y_test_pred_boot = y_test_pred_xgb1[boot_indices]
        y_test_proba_boot = y_test_proba_xgb1[boot_indices]

        metrics = get_mlflow_confidence_metrics(
            y_test_boot, y_test_pred_boot, y_test_proba_boot, 
            y_train_true, y_train_pred_xgb1, y_train_proba_xgb1, 
            xgb1.classes_
        )
        all_metrics_1.append(metrics)

        report_dict = classification_report(
            y_test_boot, y_test_pred_boot, target_names=target_names, 
            output_dict=True, zero_division=0
        )
        
        flat_report = {}
        for label, class_metrics in report_dict.items():
            if isinstance(class_metrics, dict):
                for metric_name, val in class_metrics.items():
                    flat_report[f"{label}_{metric_name}"] = val
            else:
                flat_report[f"{label}"] = class_metrics
        all_class_reports_1.append(flat_report)

    pd.DataFrame(all_metrics_1).quantile([0.025, 0.50, 0.975]).T.to_csv("./plots/xgb1_overall_metrics_quantiles.csv")
    pd.DataFrame(all_class_reports_1).quantile([0.025, 0.50, 0.975]).T.to_csv("./plots/xgb1_classification_report_quantiles.csv")
    mlflow.log_artifact("./plots/xgb1_overall_metrics_quantiles.csv", artifact_path="metrics")
    mlflow.log_artifact("./plots/xgb1_classification_report_quantiles.csv", artifact_path="metrics")


# =========================================================
# 4. XGBOOST 2 - Penalized \lambda=0.5 (Paired Bootstrap)
# =========================================================
with mlflow.start_run(run_name="xgboost_2_paired"):
    mlflow.set_tag("model_type", "xgboost 2 paired bootstrapped")
    mlflow.set_tag("data type", "24 hour transaction distribution")

    all_metrics_2 = []
    all_class_reports_2 = []

    for boot_indices in shared_bootstrap_indices:
        y_test_boot = y_test_true[boot_indices]
        y_test_pred_boot = y_test_pred_xgb2[boot_indices]
        y_test_proba_boot = y_test_proba_xgb2[boot_indices]

        metrics = get_mlflow_confidence_metrics(
            y_test_boot, y_test_pred_boot, y_test_proba_boot, 
            y_train_true, y_train_pred_xgb2, y_train_proba_xgb2, 
            xgb2.classes_
        )
        all_metrics_2.append(metrics)

        report_dict = classification_report(
            y_test_boot, y_test_pred_boot, target_names=target_names, 
            output_dict=True, zero_division=0
        )
        
        flat_report = {}
        for label, class_metrics in report_dict.items():
            if isinstance(class_metrics, dict):
                for metric_name, val in class_metrics.items():
                    flat_report[f"{label}_{metric_name}"] = val
            else:
                flat_report[f"{label}"] = class_metrics
        all_class_reports_2.append(flat_report)

    pd.DataFrame(all_metrics_2).quantile([0.025, 0.50, 0.975]).T.to_csv("./plots/xgb2_overall_metrics_quantiles.csv")
    pd.DataFrame(all_class_reports_2).quantile([0.025, 0.50, 0.975]).T.to_csv("./plots/xgb2_classification_report_quantiles.csv")
    mlflow.log_artifact("./plots/xgb2_overall_metrics_quantiles.csv", artifact_path="metrics")
    mlflow.log_artifact("./plots/xgb2_classification_report_quantiles.csv", artifact_path="metrics")

🏃 View run logistic_regression_paired at: http://127.0.0.1:5000/#/experiments/2/runs/ffd23a2523d14474940298e2c64028e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run xgboost_1_paired at: http://127.0.0.1:5000/#/experiments/2/runs/c1846d27c237436b948521b1d8fe12c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run xgboost_2_paired at: http://127.0.0.1:5000/#/experiments/2/runs/55f89fbf477e41329a92d1890f2364c7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### 6: Model Deployment

Serializes the final production model (XGBoost 2) and the target LabelEncoder into a binary .pkl artifact. This bundle acts as the inference engine for the interactive Streamlit application.

In [40]:
import pickle

# Bundle your trained model, label encoder, and feature names together
model_bundle = {
    "model": xgb2,
    "encoder": le,
}

with open("timezone_model.pkl", "wb") as f:
  pickle.dump(model_bundle, f)

print("Model successfully pickled!")

Model successfully pickled!
